# Deploy AgentCore Memory for Betting & Gaming Demo

Creates an AgentCore Memory resource with three long-term strategies:
- **Semantic** — extracts facts (e.g. "John placed a bet on Dortmund")
- **Summary** — generates session summaries
- **User Preference** — captures preferences (e.g. "prefers football betting")

Short-term memory is included by default for within-session conversation persistence.

<hr>

In [ ]:
!pip install bedrock-agentcore --quiet

## Step 1: Create Memory Resource\n<hr>

In [ ]:
from bedrock_agentcore.memory import MemoryClient

REGION = "us-east-1"
MEMORY_NAME = "BettingDemoMemory"

client = MemoryClient(region_name=REGION)

# Check if memory already exists
existing = client.list_memories()
memory_id = None
for m in existing:
    if m.get("name") == MEMORY_NAME:
        memory_id = m["id"]
        print(f"Memory already exists: {memory_id}")
        break

if not memory_id:
    print(f"Creating memory: {MEMORY_NAME}")
    memory = client.create_memory_and_wait(
        name=MEMORY_NAME,
        description="Memory for betting and gaming demo - supports both betting assistant and customer support scenarios",
        strategies=[
            {
                "semanticMemoryStrategy": {
                    "name": "FactExtractor",
                    "namespaceTemplates": ["/facts/{actorId}/"]
                }
            },
            {
                "summaryMemoryStrategy": {
                    "name": "SessionSummarizer",
                    "namespaceTemplates": ["/summaries/{actorId}/{sessionId}/"]
                }
            },
            {
                "userPreferenceMemoryStrategy": {
                    "name": "PreferenceLearner",
                    "namespaceTemplates": ["/preferences/{actorId}/"]
                }
            },
        ]
    )
    memory_id = memory["id"]
    print(f"Created memory: {memory_id}")

print(f"\nMemory ID: {memory_id}")
print("Save this ID — you need it in the agent code.")

## Step 2: Verify Memory Resource\n<hr>

In [ ]:
# Verify the memory resource
memory_info = client.get_memory(memory_id=memory_id)
print(f"Memory Name: {memory_info.get('name')}")
print(f"Memory ID: {memory_info.get('id')}")
print(f"Status: {memory_info.get('status')}")

strategies = memory_info.get('strategies', [])
print(f"\nStrategies ({len(strategies)}):")
for s in strategies:
    for key, val in s.items():
        print(f"  - {key}: {val.get('name')}")

## Step 3: Test Short-Term Memory\n<hr>

In [ ]:
from datetime import datetime
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager
from strands import Agent
from strands.models import BedrockModel

# Test with a simple agent
test_session_id = f"test-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"
test_actor_id = "test-user"

config = AgentCoreMemoryConfig(
    memory_id=memory_id,
    session_id=test_session_id,
    actor_id=test_actor_id,
)

with AgentCoreMemorySessionManager(config, region_name=REGION) as session_manager:
    model = BedrockModel(model_id="global.anthropic.claude-opus-4-6-v1")
    agent = Agent(
        model=model,
        system_prompt="You are a helpful assistant. Be brief.",
        session_manager=session_manager,
    )
    
    response = agent("My name is John and I love football betting.")
    print(f"Response 1: {response}")
    
    response = agent("What sport do I like?")
    print(f"Response 2: {response}")

print("\nShort-term memory test complete!")

## Step 4: Test Long-Term Memory (Cross-Session)\n
Wait ~30 seconds after Step 3 for the long-term strategies to process, then run this with a **new session ID** but the **same actor ID**.\n<hr>

In [ ]:
# New session, same actor — should recall preferences from long-term memory
new_session_id = f"test-session-2-{datetime.now().strftime('%Y%m%d%H%M%S')}"

config2 = AgentCoreMemoryConfig(
    memory_id=memory_id,
    session_id=new_session_id,
    actor_id=test_actor_id,  # same actor as before
)

with AgentCoreMemorySessionManager(config2, region_name=REGION) as session_manager2:
    agent2 = Agent(
        model=model,
        system_prompt="You are a helpful assistant. Use everything you know about the user.",
        session_manager=session_manager2,
    )
    
    response = agent2("Hi, I'm back. What do you remember about me?")
    print(f"Response: {response}")

print("\nLong-term memory test complete!")

## Step 5: Inspect Memory Records\n<hr>

In [ ]:
# List long-term memory records for the test actor
print("--- Long-term memory records ---")
records = client.list_memory_records(memory_id=memory_id)
for r in records.get("memoryRecords", []):
    print(f"  [{r.get('namespace', 'unknown')}] {r.get('content', {}).get('text', 'N/A')[:100]}")

if not records.get("memoryRecords"):
    print("  No records yet — long-term extraction may still be processing.")
    print("  Wait 30-60 seconds and re-run this cell.")

## Clean Up (run only when done with the demo)\n<hr>

In [ ]:
# # Uncomment to delete the memory resource
# client.delete_memory(memory_id=memory_id)
# print(f"Deleted memory: {memory_id}")